# Final Project Model
Our data comes from the [DPD Incidents](https://live-durhamnc.opendata.arcgis.com/documents/7132216432df4957830593359b0c4030/about) dataset.

## Data Cleaning
The code block below cleans the data.
1. Load data from excel sheet.
2. Filter for only 2023 and 2023 incidents.
3. Remove "(blank)" entries from all columns except for weapons.
4. Drop incidents with a status of unfounded or closed.
5. Aggregate data by concatenating, choosing the first value, or taking the mean.

In [1]:
import numpy as np 
import pandas as pd
from pathlib import Path

downloads_path = Path("C:/Users/aidan/Downloads")

# df_incidents = pd.read_excel(downloads_path / "dpd_incidents_(ucr_nibrs_reporting).xlsx")
df_incidents = pd.read_parquet("./dataframes/incident.parquet")

In [2]:
year = df_incidents["Report Date"].str.split("/").str[-1].str.strip()
# df_23_24 = df_incidents[(year == "2023") | (year == "2024")].reset_index(drop=True)
df_23_24 = pd.read_parquet("./dataframes/23_24.parquet")

for col in df_23_24.columns:
    if col != "Weapon":
        df_23_24 = df_23_24[df_23_24[col].astype(str).str.strip() != "(blank)"].reset_index(drop=True)

# df_cleaned = df_23_24[(df_23_24["Status"]!="Unfounded") & (df_23_24["Status"]!="Closed (Non-Criminal)")].reset_index(drop=True)
df_cleaned = pd.read_parquet("./dataframes/cleaned.parquet")

agg_dict = {
    "UCR Code": lambda x: ", ".join(x.astype(str)),
    "ATT/COM": lambda x: ", ".join(x.astype(str)),
    "Sequence": lambda x: ", ".join(x.astype(str)),
    "Weapon": lambda x: ", ".join(x.astype(str)),
    "Offense": lambda x: ", ".join(x.astype(str)),
    "X": "mean",
    "Y": "mean", 
    "Report Date": "first",
    "Report Time": "first",
    "Status": "first",
    "Address": "first",
    "District": "first",
    "Beat": "first",
    "Tract": "first",
    "Premise": "first"
}

# df_aggregated = df_cleaned.groupby("Case Number").agg(agg_dict).reset_index(drop=True)
df_aggregated = pd.read_parquet("./dataframes/aggregated.parquet")

# df_incidents.to_parquet("./dataframes/incident.parquet")
# df_23_24.to_parquet("./dataframes/23_24.parquet")
# df_cleaned.to_parquet("./dataframes/cleaned.parquet")
# df_aggregated.to_parquet("./dataframes/aggregated.parquet")

## Modeling
Make a SVM that predicts location (X and Y), time (month of year), and severity (UCR Code) based on other features. 

Prior to creating the model, the following steps were completed:
1. Extract the month from the report date.
2. One-hot encode everything else.
3. Drop case number, address, and report time. 
4. Aggregate the dataframe.

In [3]:
from sklearn.preprocessing import OneHotEncoder

df_cleaned["Report Month"] = df_cleaned["Report Date"].astype(str).str.split("/").str[0]
df_svm = df_cleaned.drop(["Address", "Report Date", "Report Time"], axis=1).reset_index(drop=True)
df_svm = df_svm.dropna().reset_index(drop=True)

to_encode = ["Status", "Sequence", "ATT/COM", "UCR Code", "Offense", "District", "Beat", "Tract", "Premise", "Weapon", "Report Month"]
ohe = OneHotEncoder(categories="auto", sparse_output=False)
oh_encoded = ohe.fit_transform(df_svm[to_encode])

columns = []
for col, cats in zip(to_encode, ohe.categories_):
    for cat in cats:
        columns.append(f"{col}_{cat}") 

df_ohe = pd.DataFrame(oh_encoded, columns=columns)
df_svm_ohe = pd.concat([df_svm, df_ohe], axis=1)
df_svm = df_svm_ohe.drop(to_encode, axis=1).reset_index(drop=True)

for col in df_svm.columns:
    df_svm[col] = df_svm[col].astype(int)

In [4]:
agg_dict = {}
for col in columns:
    agg_dict[col] = lambda x: 1 if (x == 1).any() else 0

agg_dict["X"] = "mean"
agg_dict["Y"] = "mean"

# df_svm_agg = df_svm.groupby("Case Number").agg(agg_dict).reset_index(drop=True)

# df_svm.to_parquet("./dataframes/svm.parquet")
# df_svm_agg.to_parquet("./dataframes/svm_agg.parquet")

We use MultiOutputRegressor to make an SVM for the targets. Since X and Y are predicted and UCR Code and Month are classified we use SVR and SVC.

In [ ]:
import numpy as np 
import pandas as pd
import pickle
from sklearn.svm import SVC, SVR
from sklearn.multioutput import MultiOutputClassifier, MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler

df_svm = pd.read_parquet("./dataframes/svm.parquet")
df_svm_agg = pd.read_parquet("./dataframes/svm_agg.parquet")

df_agg = df_svm

to_encode = ["Status", "Sequence", "ATT/COM", "UCR Code", "Offense", "District", "Beat", "Tract", "Premise", "Weapon", "Report Month"]

y_col = []
for col, cats in zip(to_encode, ohe.categories_):
    if col in ["UCR Code", "Report Month"]:
        for cat in cats:
            y_col.append(f"{col}_{cat}") 

X = df_agg[list(set(df_agg.columns) - set(y_col))]
y = df_agg[y_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

valid_columns = []
for col in y_col:
    if y_train[col].nunique() > 1:
        valid_columns.append(col)
    else:
        print(f"Skipping column {col} - only has 1 unique value")

y_train_valid = y_train[valid_columns]
y_test_valid = y_test[valid_columns]

svc = SVC(kernel='rbf', probability=True, C=1.0)
mor_svc = MultiOutputClassifier(svc, n_jobs=-1)

fitted_svc = mor_svc.fit(X_train_scaled, y_train_valid)

y_pred = mor_svc.predict(X_test_scaled)

accuracy = accuracy_score(y_test_valid, y_pred)
print(f"Overall accuracy: {accuracy:.4f}")

for i, col in enumerate(y_col):
    print(f"Accuracy for {col}: {accuracy_score(y_test_valid.iloc[:, i], y_pred[:, i]):.4f}")

filename = 'svc_model.pkl'
pickle.dump(mor_svc, open(filename, 'wb'))

KeyboardInterrupt: 

In [ ]:
y_col = []
for col, cats in zip(to_encode, ohe.categories_):
    if col in ["X", "Y"]:
        for cat in cats:
            y_col.append(f"{col}_{cat}") 

X = df_agg[list(set(df_agg.columns) - set(y_col))]
y = df_agg[y_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

svr = SVR(kernel='rbf', C=1.0, epsilon=0.1)
mor_svr = MultiOutputRegressor(svr, n_jobs=-1)
fitted_svr = mor_svr.fit(X_train_scaled, y_train)

y_pred = mor_svr.predict(X_test_scaled)

from sklearn.metrics import mean_squared_error, r2_score
mse = mean_squared_error(y_test, y_pred, multioutput='raw_values')
r2 = r2_score(y_test, y_pred, multioutput='raw_values')

for i, col in enumerate(y_col):
   print(f"{col}: MSE = {mse[i]:.4f}, R² = {r2[i]:.4f}")

print(f"Average MSE: {np.mean(mse):.4f}")
print(f"Average R²: {np.mean(r2):.4f}")

filename = 'svr_model.pkl'
pickle.dump(mor_svr, open(filename, 'wb'))

ValueError: at least one array or dtype is required